# Solutions — Schema and Types (Hard 10)

In [ ]:
from pyspark.sql import functions as F, types as T

transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")
orders       = spark.table("samples.tpch.orders")


## Solution 1 — Inspect and Compare Schemas

In [ ]:
rows = [(f.name, str(f.dataType), f.nullable) for f in transactions.schema]
result_1 = spark.createDataFrame(rows, ["column_name", "data_type", "nullable"])


## Solution 2 — Define an Explicit StructType Schema

In [ ]:
result_2 = (
    transactions
    .withColumn("transactionID",        F.col("transactionID").cast(T.LongType()))
    .withColumn("totalPrice_as_double", F.col("totalPrice").cast(T.DoubleType()))
    .select("transactionID", "product", "totalPrice_as_double")
)


## Solution 3 — Create a Nested Struct Column

In [ ]:
result_3 = (
    customers
    .withColumn("full_name",
        F.concat_ws(" ", F.col("firstName"), F.col("lastName"))
    )
    .withColumn("address_struct",
        F.struct(
            F.col("city").alias("city"),
            F.col("state").alias("state"),
            F.col("country").alias("country"),
        )
    )
    .withColumn("country", F.col("address_struct.country"))
    .select("customerID", "full_name", "address_struct", "country")
)


## Solution 4 — Schema from a DDL String

In [ ]:
ddl = "orderkey BIGINT, custkey BIGINT, status STRING, totalprice DOUBLE, orderdate DATE"
ddl_schema = T.StructType.fromDDL(ddl)

result_4 = (
    orders
    .select(
        F.col("o_orderkey").cast(T.LongType()).alias("orderkey"),
        F.col("o_custkey").cast(T.LongType()).alias("custkey"),
        F.col("o_orderstatus").alias("status"),
        F.col("o_totalprice").cast(T.DoubleType()).alias("totalprice"),
        F.col("o_orderdate").cast(T.DateType()).alias("orderdate"),
    )
)


## Solution 5 — ArrayType and MapType Columns

In [ ]:
payment_agg = (
    transactions
    .groupBy("franchiseID", "paymentMethod")
    .agg(F.count("*").alias("pm_count"))
    .groupBy("franchiseID")
    .agg(
        F.map_from_entries(
            F.collect_list(F.struct(F.col("paymentMethod"), F.col("pm_count")))
        ).alias("payment_count_map")
    )
)

result_5 = (
    transactions
    .groupBy("franchiseID")
    .agg(F.collect_list("product").alias("products_array"))
    .join(payment_agg, on="franchiseID")
    .select("franchiseID", "products_array", "payment_count_map")
)


## Solution 6 — Schema Mismatch: Safe Casting

In [ ]:
result_6 = (
    customers
    .withColumn("zip_as_string", F.col("customerID").cast(T.StringType()))
    .withColumn("zip_as_int",
        F.when(F.col("zip_as_string").rlike(r"^\d+$"),
               F.col("zip_as_string").cast(T.IntegerType()))
    )
    .select("customerID", "zip_as_string", "zip_as_int")
)


## Solution 7 — Compare Schemas of Two DataFrames

In [ ]:
def compare_schemas(df1, df2):
    s1 = {f.name.lower(): str(f.dataType) for f in df1.schema}
    s2 = {f.name.lower(): str(f.dataType) for f in df2.schema}
    rows = []
    for col in set(s1) | set(s2):
        t1 = s1.get(col)
        t2 = s2.get(col)
        if col in s1 and col not in s2:
            rows.append((col, "only_in_transactions", t1, None))
        elif col not in s1 and col in s2:
            rows.append((col, "only_in_customers", None, t2))
        elif t1 != t2:
            rows.append((col, "type_mismatch", t1, t2))
    return rows

rows = compare_schemas(transactions, customers)
result_7 = spark.createDataFrame(
    rows,
    ["column_name", "present_in", "type_in_transactions", "type_in_customers"]
)


## Solution 8 — Schema Evolution Simulation

In [ ]:
evolved = (
    transactions
    .withColumn("discount_pct",     F.lit(0.0))           # ADD
    .withColumnRenamed("paymentMethod", "payment_method_v2")  # RENAME
    .drop("cardNumber")                                    # DROP
)

change_log = [
    ("ADD",    "discount_pct",       "Added as 0.0 literal"),
    ("RENAME", "paymentMethod",      "Renamed to payment_method_v2"),
    ("DROP",   "cardNumber",         "Removed — not needed downstream"),
]

result_8 = spark.createDataFrame(change_log, ["change_type", "column_name", "detail"])
